**Завдання 6. Ноутбук 05 — історична таблиця фактів і злиття**

In [1]:
!pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

In [2]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "project-nbu"   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-nbu


**Завдання 6.1.** Прочитайте з nbu_raw.raw_rates рядки тільки за сьогодні.
WHERE business_date = CURRENT_DATE()

In [4]:
# --- ЗАВДАННЯ 6.1: Читання строк за останній наявний день ---

# Запит автоматично знаходить максимальну дату в базі та бере рядки тільки за неї
query_today = f"""
SELECT ingested_at, business_date, payload
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
WHERE business_date = (SELECT MAX(business_date) FROM `{PROJECT_ID}.nbu_raw.raw_rates`)
"""

# Завантажуємо цей срез даних у DataFrame
df_today_bronze = client.query(query_today).to_dataframe()

print(f"✅ Дані успішно завантажені!")
print(f"Робоча дата (business_date): {df_today_bronze['business_date'].iloc[0]}")
print(f"Кількість строк у вибірці: {len(df_today_bronze)}")


✅ Дані успішно завантажені!
Робоча дата (business_date): 2026-08-25
Кількість строк у вибірці: 90


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 6.2.** Розгорніть payload, нормалізуйте значення й приберіть дублікати усередині свого набору: зерно таблиці — одна валюта на одну дату, тому на пару business_date + currency_code має лишитися один рядок (найсвіжіший за ingested_at).

In [5]:
# --- ЗАВДАННЯ 6.2: Розгортання, нормалізація та внутрішня дедуплікація ---
import json

# 1. Розгортаємо JSON-текст із payload у колонки
df_parsed_today = pd.json_normalize(df_today_bronze["payload"].map(json.loads))

# Додаємо системні поля часу та бізнес-дати з початкового набору
df_parsed_today["ingested_at"] = df_today_bronze["ingested_at"]
df_parsed_today["business_date"] = df_today_bronze["business_date"]

# 2. Нормалізуємо значення полів
df_parsed_today["cc"] = df_parsed_today["cc"].str.strip().str.upper()
df_parsed_today["txt"] = df_parsed_today["txt"].str.strip()
df_parsed_today["r030"] = df_parsed_today["r030"].astype("Int64")
df_parsed_today["rate"] = df_parsed_today["rate"].astype(float)

# 3. Прибираємо дублікати всередині сьогоднішнього набору даних
# Сортуємо за ingested_at, щоб найсвіжіші запуски опинилися внизу
df_today_clean = (
    df_parsed_today.sort_values("ingested_at")
    .drop_duplicates(subset=["business_date", "cc"], keep="last")
    .copy()
)

# Перевірка результату дедуплікації
print(f"Початкова кількість рядків за сьогодні: {len(df_parsed_today)}")
print(f"Кількість рядків після дедуплікації (зерно таблиці): {len(df_today_clean)}")
print("\nФрагмент очищених сьогоднішніх даних:")
print(df_today_clean[["business_date", "cc", "rate", "ingested_at"]].head(3))

Початкова кількість рядків за сьогодні: 90
Кількість рядків після дедуплікації (зерно таблиці): 45

Фрагмент очищених сьогоднішніх даних:
   business_date   cc     rate                      ingested_at
75    2026-08-25  EGP   0.8797 2026-08-24 13:46:45.990685+00:00
74    2026-08-25  TND  15.3888 2026-08-24 13:46:45.990685+00:00
73    2026-08-25  AED  12.1716 2026-08-24 13:46:45.990685+00:00


**Завдання 6.3.** Підставте ключі вимірів: date_key — з business_date у форматі YYYYMMDD, currency_key — приєднанням nbu_dwh.dim_currency, з -1 для ненайдених валют.
Отримати на виході: date_key, currency_key, rate, business_date, dw_load_ts

In [6]:
# --- ЗАВДАННЯ 6.3: Підстановка ключів та формування фінальної структури фактів ---

# 1. Формуємо date_key з business_date у форматі YYYYMMDD
# Перетворюємо в рядок YYYYMMDD, а потім у ціле число
df_today_clean["date_key"] = pd.to_datetime(df_today_clean["business_date"]).dt.strftime("%Y%m%d").astype(int)

# 2. Читаємо актуальний вимір валют із шару Gold для отримання ключів
query_dim_currency = f"SELECT currency_key, currency_code FROM `{PROJECT_ID}.nbu_dwh.dim_currency`"
df_dim_currency = client.query(query_dim_currency).to_dataframe()

# 3. Приєднуємо вимір валют через Left Join
facts_joined = pd.merge(
    df_today_clean,
    df_dim_currency,
    left_on="cc",
    right_on="currency_code",
    how="left"
)

# 4. Заповнюємо пропущені ключі валют значенням -1
facts_joined["currency_key"] = facts_joined["currency_key"].fillna(-1).astype(int)

# 5. Додаємо системне поле мітки часу завантаження в UTC
facts_joined["dw_load_ts"] = pd.Timestamp.now(tz="UTC")

# 6. Відбираємо тільки необхідні колонки у строго заданому порядку
target_cols = ["date_key", "currency_key", "rate", "business_date", "dw_load_ts"]
df_facts_today = facts_joined[target_cols].copy()

# Перевірка результату перед злиттям
print("--- Схема полів поточної порції фактів ---")
print(df_facts_today.dtypes)

print("\n--- Перші 3 рядки підготовлених фактів за сьогодні ---")
print(df_facts_today.head(3))

--- Схема полів поточної порції фактів ---
date_key                       int64
currency_key                   int64
rate                         float64
business_date                 dbdate
dw_load_ts       datetime64[us, UTC]
dtype: object

--- Перші 3 рядки підготовлених фактів за сьогодні ---
   date_key  currency_key     rate business_date  \
0  20260825            11   0.8797    2026-08-25   
1  20260825            36  15.3888    2026-08-25   
2  20260825             1  12.1716    2026-08-25   

                        dw_load_ts  
0 2026-08-27 14:50:36.927890+00:00  
1 2026-08-27 14:50:36.927890+00:00  
2 2026-08-27 14:50:36.927890+00:00  


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 6.4.** Створіть таблицю nbu_dwh.fact_exchange_rate, якщо її ще немає: схема з пункту 6.3, партиціювання за business_date, кластеризація за currency_key.

In [7]:
# --- ЗАВДАННЯ 6.4: Створення партиційованої та кластеризованої таблиці фактів ---

FACT_ID = f"{PROJECT_ID}.nbu_dwh.fact_exchange_rate"

# Описуємо схему таблиці на основі полів із пункту 6.3
fact_schema = [
    bigquery.SchemaField("date_key", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("currency_key", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("rate", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("business_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("dw_load_ts", "TIMESTAMP", mode="REQUIRED"),
]

# Створюємо об'єкт таблиці
table = bigquery.Table(FACT_ID, schema=fact_schema)

# Задаємо щоденне партиціювання за полем business_date
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="business_date"
)

# Задаємо кластеризацію за полем currency_key
table.clustering_fields = ["currency_key"]

# Створюємо таблицю (exists_ok=True захищає від помилок при повторному запуску)
client.create_table(table, exists_ok=True)

print(f"✅ Таблицю {FACT_ID} успішно створено з партиціюванням та кластеризацією!")

✅ Таблицю project-nbu.nbu_dwh.fact_exchange_rate успішно створено з партиціюванням та кластеризацією!


**Завдання 6.5.** Реалізуйте злиття: рядки за сьогодні мають бути оновлені або додані, рядки за попередні дні — залишитися недоторканими. Оберіть один із двох способів, описаних нижче.

In [13]:
# --- ЗАВДАННЯ 6.5: Фінальний варіант запису в партицію зі схемою ---

FACT_TABLE_ID = f"{PROJECT_ID}.nbu_dwh.fact_exchange_rate"

# Дізнаємося робочу дату
working_date = df_facts_today["business_date"].iloc[0]
partition_suffix = pd.to_datetime(working_date).strftime("%Y%m%d")

# Налаштовуємо конфігурацію: передаємо правильну схему (schema=fact_schema) 
# та режим перезапису конкретної партиції
job_config = bigquery.LoadJobConfig(
    schema=fact_schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Записуємо дані у партицію
target_partition = f"{FACT_TABLE_ID}${partition_suffix}"
job = client.load_table_from_dataframe(df_facts_today, target_partition, job_config=job_config)
job.result()  # Очікуємо завершення

print(f"✅ Дані успішно записані в партицію {target_partition}!")
print("Схема полів повністю валідована, обмеження Sandbox обійдено!")

✅ Дані успішно записані в партицію project-nbu.nbu_dwh.fact_exchange_rate$20260825!
Схема полів повністю валідована, обмеження Sandbox обійдено!


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


**Завдання 6.6.** Перевірте й виведіть результат трьома запитами:
зерно унікальне — немає жодної пари date_key + currency_key, що трапляється двічі;
кількість різних business_date у таблиці;
загальна кількість рядків.

In [14]:
# --- ЗАВДАННЯ 6.6: Три перевірочні запити для валідації таблиці фактів ---

print("=== Перевірка 1: Пошук дублікатів по зерну (має повернути порожній DataFrame) ===")
query_grain = f"""
SELECT date_key, currency_key, COUNT(*) as row_cnt
FROM `{PROJECT_ID}.nbu_dwh.fact_exchange_rate`
GROUP BY date_key, currency_key
HAVING row_cnt > 1
"""
df_grain_check = client.query(query_grain).to_dataframe()
print(df_grain_check)
print(f"Зерно унікальне: {len(df_grain_check) == 0}")

print("\n=== Перевірка 2: Кількість різних business_date у таблиці ===")
query_dates = f"""
SELECT COUNT(DISTINCT business_date) as unique_dates_cnt
FROM `{PROJECT_ID}.nbu_dwh.fact_exchange_rate`
"""
df_dates_check = client.query(query_dates).to_dataframe()
print(df_dates_check)

print("\n=== Перевірка 3: Загальна кількість рядків у таблиці фактів ===")
query_rows = f"""
SELECT COUNT(*) as total_rows_cnt
FROM `{PROJECT_ID}.nbu_dwh.fact_exchange_rate`
"""
df_rows_check = client.query(query_rows).to_dataframe()
print(df_rows_check)

=== Перевірка 1: Пошук дублікатів по зерну (має повернути порожній DataFrame) ===
Empty DataFrame
Columns: [date_key, currency_key, row_cnt]
Index: []
Зерно унікальне: True

=== Перевірка 2: Кількість різних business_date у таблиці ===
   unique_dates_cnt
0                 1

=== Перевірка 3: Загальна кількість рядків у таблиці фактів ===
   total_rows_cnt
0              45


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


**Завдання 6.7.** Запустіть ноутбук двічі підряд і порівняйте результати пункту 6.6: обидва числа мають бути однаковими. Якщо кількість рядків зросла — злиття зроблено неправильно, воно додало дублікати замість оновлення.

*Ідемпотентність конвеєра досягнута: До перезапуску в таблиці було 45 рядків, і після повторного виконання коду їхня кількість залишилася рівно 45. Це означає, що повторні запуски системи безпечні й не засмічують базу даних.Дублікати повністю відсутні: Зерно унікальне: True підтверджує, що комбінація дата + валюта не повторюється двічі. Замість додавання копій, система коректно оновила існуючі записи (принцип Upsert).Обмеження Sandbox успішно обійти: Стратегія перезапису конкретної щоденної партиції (fact_exchange_rate$YYYYMMDD) за допомогою Pandas спрацювала як повна інженерна альтернатива SQL-команді MERGE, яка заблокована у безкоштовному тарифі Google Cloud.*